# Set up RelationalAI (PyRel) and load the sample data

This notebook runs locally. It connects to Snowflake with your `raiconfig.yaml`, uploads the CSV files from `assets/data/` and loads the tables, then runs a first PyRel query. Afterwards, continue with `02_supplier_risk_to_plan.ipynb`.

**Prerequisites:**
1. An administrator has completed [Set up RelationalAI in Snowflake](https://docs.relational.ai/get-started/set-up-relationalai-in-snowflake) and granted you the role `RAI_DEVELOPER`.
2. An administrator has run `assets/scripts/setup.sql`, which creates the warehouse `RAI_QS_WH` and the schema `RAI_QUICKSTART.SUPPLY_CHAIN` for `RAI_DEVELOPER`.
3. A Python 3.10–3.14 environment with `pip install "relationalai>=1.30.1,<2.0" jupyter`, registered as the kernel **RAI supply chain** (see the guide's Setup section). Select that kernel for this notebook.
4. `raiconfig.yaml` is saved in the guide folder (two levels above this notebook), with `warehouse: RAI_QS_WH` and `role: RAI_DEVELOPER`.

In [ ]:
# 1. Connect. create_config() reads raiconfig.yaml from this folder (or a parent folder).
#    With OAuth browser sign-in, a browser window opens the first time. Prints no secrets.
import importlib.metadata as md
import sys

from relationalai.config import create_config

cfg = create_config()
session = cfg.get_session()   # a Snowpark session, used below for SQL and file upload
role, wh = session.sql("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE()").collect()[0]

LLM = "claude-sonnet-4-5"     # the model notebook 02 uses; change it there too if it isn't available


def cortex_ok(model):
    """A one-token AI_COMPLETE call: checks Cortex access for this role and the model in this region."""
    try:
        session.sql("SELECT AI_COMPLETE(?, 'Reply with OK.', {'max_tokens': 1})", params=[model]).collect()
        return True
    except Exception as e:
        print(f"     AI_COMPLETE failed: {str(e).splitlines()[0]}")
        return False


checks = {
    "Python >= 3.10": sys.version_info[:2] >= (3, 10),
    "Role RAI_DEVELOPER": role == "RAI_DEVELOPER",
    "Warehouse RAI_QS_WH": wh == "RAI_QS_WH",
    "RelationalAI app visible": len(session.sql("SHOW APPLICATIONS LIKE 'RELATIONALAI'").collect()) == 1,
    "Schema from setup.sql": len(session.sql("SHOW SCHEMAS LIKE 'SUPPLY_CHAIN' IN DATABASE RAI_QUICKSTART").collect()) == 1,
    f"Cortex AI_COMPLETE with {LLM}": cortex_ok(LLM),
}
print(f"Python {sys.version.split()[0]} | relationalai {md.version('relationalai')} | role {role} | warehouse {wh}")
for name, ok in checks.items():
    print(("OK  " if ok else "FAIL"), name)

## 2. Load the sample data

Upload the CSV files from `assets/data/` to the internal stage `DATA_STAGE`, then run `assets/scripts/load_data.sql`. The script creates eleven tables with `CHANGE_TRACKING = TRUE`: RelationalAI keeps its copy of source tables in sync through change data capture (CDC). Safe to re-run.

In [ ]:
# 2. Upload data/*.csv and run load_data.sql, one statement at a time.
from pathlib import Path

assets = Path.cwd().parent   # assets/, one level above notebooks/
session.sql("CREATE STAGE IF NOT EXISTS RAI_QUICKSTART.SUPPLY_CHAIN.DATA_STAGE").collect()
for csv in sorted((assets / "data").glob("*.csv")):
    session.file.put(str(csv), "@RAI_QUICKSTART.SUPPLY_CHAIN.DATA_STAGE", auto_compress=False, overwrite=True)

script = (assets / "scripts" / "load_data.sql").read_text()
code = "\n".join(line for line in script.splitlines() if not line.lstrip().startswith("--"))
statements = [s.strip() for s in code.split(";") if s.strip()]
for statement in statements[:-1]:
    session.sql(statement).collect()

counts = session.sql(statements[-1]).to_pandas()   # the last statement checks the row counts
print(counts.to_string(index=False))
expected = {"PARTS": 315, "PRODUCTS": 20, "BOM": 469, "SUBSTITUTES": 60, "SUPPLIERS": 80, "SUPPLIER_PARTS": 483,
            "SUPPLIER_DEPENDENCIES": 81, "QUALIFIED_SUPPLIERS": 1, "INVENTORY": 315, "CUSTOMERS": 100, "ORDERS": 1500}
assert dict(zip(counts.TABLE_NAME, counts.ROW_COUNT)) == expected

In [ ]:
# 3. A first, tiny model and query: which tier-1 suppliers supply the temperature sensor P-141?
#    The first query takes several minutes: RelationalAI starts an engine and indexes the tables.
from relationalai.semantics import Model, String

model = Model("SetupCheck", config=cfg)
Supplier = model.Concept("Supplier", identify_by={"id": String})
Supplier.region = model.Property(f"{Supplier} is in region {String:region}")
Supplier.supplies = model.Relationship(f"{Supplier} supplies part {String:part_id}")

suppliers = model.Table("RAI_QUICKSTART.SUPPLY_CHAIN.SUPPLIERS")
supplier_parts = model.Table("RAI_QUICKSTART.SUPPLY_CHAIN.SUPPLIER_PARTS")
model.define(s := Supplier.new(id=suppliers.SUPPLIER_ID), s.region(suppliers.REGION))
model.define(Supplier.lookup(id=supplier_parts.SUPPLIER_ID).supplies(supplier_parts.PART_ID))

result = model.where(Supplier.supplies("P-141")).select(
    Supplier.id.alias("supplier"), Supplier.region.alias("region")).to_df().sort_values("supplier")
print(result)
assert list(result["supplier"]) == ["S-T1-02", "S-T1-05"]
print("Setup works. Continue with 02_supplier_risk_to_plan.ipynb")